In [2]:
# ── TUNING XGBoost ─────────────────────────────────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

# ── CARGA DE DATOS ─────────────────────────────────────────────────────────────
df = pd.read_csv("../dataset/heart_processed.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [3]:
# ── BASELINE (sin tuning) ──────────────────────────────────────────────────────
baseline = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)
y_proba = baseline.predict_proba(X_test)[:, 1]

print("── XGBoost Baseline ──────────────────────────────")
print(classification_report(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

── XGBoost Baseline ──────────────────────────────
              precision    recall  f1-score   support

           0       0.84      0.74      0.79        82
           1       0.81      0.88      0.85       102

    accuracy                           0.82       184
   macro avg       0.82      0.81      0.82       184
weighted avg       0.82      0.82      0.82       184

ROC AUC: 0.8845


In [4]:
# ── RANDOMIZED SEARCH ─────────────────────────────────────────────────────────
param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Mejores parámetros:", random_search.best_params_)
print(f"Mejor ROC AUC (cv): {random_search.best_score_:.4f}")

Mejores parámetros: {'subsample': 0.8, 'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
Mejor ROC AUC (cv): 0.8753


In [5]:
# ── EVALUACIÓN DEL MODELO TUNEADO ─────────────────────────────────────────────
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("── XGBoost Tuneado ──────────────────────────────")
print(classification_report(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

── XGBoost Tuneado ──────────────────────────────
              precision    recall  f1-score   support

           0       0.82      0.73      0.77        82
           1       0.80      0.87      0.84       102

    accuracy                           0.81       184
   macro avg       0.81      0.80      0.80       184
weighted avg       0.81      0.81      0.81       184

ROC AUC: 0.9088


In [6]:
# ── COMPARACIÓN BASELINE VS TUNEADO ───────────────────────────────────────────
results = {
    'Modelo': ['XGBoost Baseline', 'XGBoost Tuneado'],
    'Accuracy': [
        (baseline.predict(X_test) == y_test).mean(),
        (best_model.predict(X_test) == y_test).mean()
    ],
    'ROC AUC': [
        roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, y_proba)
    ]
}

print(pd.DataFrame(results).to_string(index=False))
print("\nMejores hiperparámetros a usar en train_model.py:")
print(random_search.best_params_)

          Modelo  Accuracy  ROC AUC
XGBoost Baseline  0.820652 0.884505
 XGBoost Tuneado  0.809783 0.908776

Mejores hiperparámetros a usar en train_model.py:
{'subsample': 0.8, 'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
